# Phase 1 — Distribution Families: Exploration Notebook

This notebook explores the five candidate distribution families for people cost modelling.
We visualize their shapes, compare properties, and generate synthetic cost data.

**Distributions:** Normal, LogNormal, Gamma, Pareto, Weibull

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.distributions import (
    NormalDist, LogNormalDist, GammaDist, ParetoDist, WeibullDist
)
from src.data_gen import (
    generate_salary_data, generate_mixed_data, inject_outliers
)

sns.set_theme(style="whitegrid", font_scale=1.1)
SEED = 42

## 1. The Distribution Zoo

Visualize all five candidate families with parameters calibrated to salary-like data.

In [ ]:
# Define distributions with salary-scale parameters
distributions = {
    "Normal": NormalDist(mu=10000, sigma=3000),
    "LogNormal": LogNormalDist(mu=9.1, sigma=0.4),
    "Gamma": GammaDist(alpha=10, beta=10/10000),
    "Weibull": WeibullDist(k=3.5, lam=11000),
}

x = np.linspace(1, 25000, 1000)

fig, ax = plt.subplots(figsize=(10, 6))
for name, dist in distributions.items():
    ax.plot(x, dist.pdf(x), label=f"{name} (mean={dist.mean():.0f})", linewidth=2)

ax.set_xlabel("Cost (R$)")
ax.set_ylabel("Density")
ax.set_title("Candidate Distribution Families — Salary Scale")
ax.legend()
ax.set_xlim(0, 25000)
plt.tight_layout()
plt.show()

## 2. Pareto Tails vs Normal Tails

The key insight: Pareto tails decay much slower than Normal tails.

In [ ]:
pareto = ParetoDist(alpha=2.5, x_m=10000)
normal = NormalDist(mu=pareto.mean(), sigma=np.sqrt(pareto.var()))

x_tail = np.linspace(10000, 100000, 1000)
survival_pareto = 1 - pareto.cdf(x_tail)
survival_normal = 1 - normal.cdf(x_tail)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(x_tail, survival_pareto, label="Pareto(2.5, 10000)", linewidth=2)
ax.semilogy(x_tail, survival_normal, label=f"Normal(mean-matched)", linewidth=2)
ax.set_xlabel("Cost threshold (R$)")
ax.set_ylabel("P(X > threshold) [log scale]")
ax.set_title("Survival Function: Pareto vs Normal (Matched Mean)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Synthetic Salary Data — Unimodal vs Bimodal

In [ ]:
# Unimodal: LogNormal
salary_unimodal = generate_salary_data(2000, seed=SEED)

# Bimodal: mixture of juniors and seniors
junior = NormalDist(mu=8000, sigma=1500)
senior = NormalDist(mu=18000, sigma=2500)
salary_bimodal = generate_mixed_data(2000, [junior, senior], [0.6, 0.4], seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(salary_unimodal, bins=50, density=True, alpha=0.7, color="steelblue")
axes[0].set_title("Unimodal: LogNormal(9.1, 0.4)")
axes[0].set_xlabel("Salary (R$)")
axes[0].set_ylabel("Density")

axes[1].hist(salary_bimodal, bins=50, density=True, alpha=0.7, color="coral")
axes[1].set_title("Bimodal: 60% Junior + 40% Senior")
axes[1].set_xlabel("Salary (R$)")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

## 4. Properties Comparison Table

In [ ]:
import pandas as pd

dists = {
    "Normal(10000, 3000)": NormalDist(mu=10000, sigma=3000),
    "LogNormal(9.1, 0.4)": LogNormalDist(mu=9.1, sigma=0.4),
    "Gamma(10, 0.001)": GammaDist(alpha=10, beta=0.001),
    "Pareto(2.5, 10000)": ParetoDist(alpha=2.5, x_m=10000),
    "Weibull(3.5, 11000)": WeibullDist(k=3.5, lam=11000),
}

rows = []
for name, d in dists.items():
    rows.append({
        "Distribution": name,
        "Mean": f"{d.mean():,.0f}",
        "Variance": f"{d.var():,.0f}",
        "Std Dev": f"{np.sqrt(d.var()):,.0f}",
        "Skewness": f"{d.skewness():.2f}",
        "Excess Kurtosis": f"{d.kurtosis_excess():.2f}",
    })

df = pd.DataFrame(rows)
df

## 5. Outlier Injection Demo

In [ ]:
clean_data = generate_salary_data(500, seed=SEED)
contaminated = inject_outliers(clean_data, fraction=0.05, multiplier=3.0, seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(clean_data, bins=40, alpha=0.7, color="steelblue")
axes[0].set_title(f"Clean Data (mean={clean_data.mean():,.0f})")
axes[0].set_xlabel("Salary (R$)")

axes[1].hist(contaminated, bins=40, alpha=0.7, color="coral")
axes[1].set_title(f"With 5% Outliers (mean={contaminated.mean():,.0f})")
axes[1].set_xlabel("Salary (R$)")

plt.tight_layout()
plt.show()

print(f"Mean shift: {contaminated.mean() - clean_data.mean():+,.0f}")
print(f"Std dev shift: {contaminated.std() - clean_data.std():+,.0f}")